In [1]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path("data/processed/incubator_prepared.csv")

df = pd.read_csv(
    INPUT_FILE,
    parse_dates=["timestamp"]
)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes.to_string())

missing_counts = df.isna().sum()
print("\nColumns with missing values:")
print(missing_counts[missing_counts > 0].to_string())

time_difference = df["timestamp"].diff()
backward_rows = time_difference < pd.Timedelta(0)

print("\nOrder checks:")
print("record_id increasing:", df["record_id"].is_monotonic_increasing)
print("timestamp chronological:", df["timestamp"].is_monotonic_increasing)
print("backward timestamp transitions:", backward_rows.sum())

print("\nSmall sample:")
print(df.head(3).to_string(index=False))

Shape: (53061, 27)

Columns:
['record_id', 'timestamp', 's1_t', 's1_h', 's1_flag', 's2_t', 's2_h', 's2_flag', 's3_t', 's3_h', 's3_flag', 's4_t', 's4_h', 's4_flag', 'avg_t', 'avg_h', 'duty_pct', 'duty_correction_pct', 'heater_on', 'humidity_fan_on', 'servo_position', 'operating_mode', 'sensor_spread_c', 'active_sensors_count', 'rotation_minutes_remaining', 'low_edge_counter', 'high_edge_counter']

Data types:
record_id                              int64
timestamp                     datetime64[us]
s1_t                                 float64
s1_h                                 float64
s1_flag                                  str
s2_t                                 float64
s2_h                                 float64
s2_flag                                  str
s3_t                                 float64
s3_h                                 float64
s3_flag                                  str
s4_t                                 float64
s4_h                                 float64
s4_

In [2]:
MAX_CONTINUOUS_GAP_MINUTES = 3

rows_before = len(df)

df["gap_minutes_from_previous"] = (
    df["timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

df["missing_minutes_before"] = (
    (df["gap_minutes_from_previous"] - 1)
    .clip(lower=0)
    .where(df["gap_minutes_from_previous"].ge(0))
    .astype("Int64")
)

In [3]:
df["is_sequence_start"] = (
    df["gap_minutes_from_previous"].isna()
    | df["gap_minutes_from_previous"].lt(0)
    | df["gap_minutes_from_previous"].gt(
        MAX_CONTINUOUS_GAP_MINUTES
    )
)

df["sequence_id"] = (
    df["is_sequence_start"]
    .cumsum()
    .astype("int32")
)

In [4]:
internal_gap = df["gap_minutes_from_previous"].mask(
    df["is_sequence_start"]
)

assert len(df) == rows_before
assert df["record_id"].is_monotonic_increasing
assert internal_gap.dropna().between(
    0, MAX_CONTINUOUS_GAP_MINUTES
).all()

sequence_summary = df.groupby("sequence_id").agg(
    start_record_id=("record_id", "first"),
    end_record_id=("record_id", "last"),
    start_time=("timestamp", "first"),
    end_time=("timestamp", "last"),
    row_count=("record_id", "size"),
)

sequence_summary["duration_minutes"] = (
    sequence_summary["end_time"]
    - sequence_summary["start_time"]
).dt.total_seconds().div(60)

print("Number of sequences:", df["sequence_id"].nunique())
print(sequence_summary.to_string())

Number of sequences: 9
             start_record_id  end_record_id          start_time            end_time  row_count  duration_minutes
sequence_id                                                                                                     
1                          1            534 2026-06-24 14:32:00 2026-06-24 16:58:00        534             146.0
2                        535           4980 2026-06-24 17:43:00 2026-06-25 13:53:00       4445            1210.0
3                       4981          18929 2026-06-25 14:03:00 2026-06-28 05:19:00      13948            3796.0
4                      18930          18930 2026-06-28 05:17:00 2026-06-28 05:17:00          1               0.0
5                      18931          28693 2026-06-28 08:08:00 2026-06-30 04:23:00       9762            2655.0
6                      28694          28694 2026-06-30 04:17:00 2026-06-30 04:17:00          1               0.0
7                      28695          48646 2026-06-30 07:26:00 2026-07-0

In [5]:
# %%
SENSORS = ("s1", "s2", "s3", "s4")
VALID_SENSOR_FLAGS = {"A", "P", "ERR"}

TEMP_PLAUSIBLE_MIN_C = 0
TEMP_PLAUSIBLE_MAX_C = 60
HUMIDITY_PLAUSIBLE_MIN_PCT = 0
HUMIDITY_PLAUSIBLE_MAX_PCT = 100

missing_count = pd.Series(0, index=df.index, dtype="int8")
err_count = pd.Series(0, index=df.index, dtype="int8")
quality_issue_count = pd.Series(0, index=df.index, dtype="int8")

quality_report = []

for sensor in SENSORS:
    temperature = df[f"{sensor}_t"]
    humidity = df[f"{sensor}_h"]
    flag = df[f"{sensor}_flag"]

    has_missing = temperature.isna() | humidity.isna()
    is_err = flag.eq("ERR")

    unexpected_missing = has_missing & ~is_err
    invalid_flag = ~flag.isin(VALID_SENSOR_FLAGS)

    implausible_value = (
        temperature.notna()
        & ~temperature.between(
            TEMP_PLAUSIBLE_MIN_C,
            TEMP_PLAUSIBLE_MAX_C,
        )
    ) | (
        humidity.notna()
        & ~humidity.between(
            HUMIDITY_PLAUSIBLE_MIN_PCT,
            HUMIDITY_PLAUSIBLE_MAX_PCT,
        )
    )

    err_with_values = (
        is_err
        & df[[f"{sensor}_t", f"{sensor}_h"]]
        .notna()
        .any(axis=1)
    )

    assert not err_with_values.any(), (
        f"{sensor}: an ERR state contains a sensor value"
    )

    quality_issue = (
        unexpected_missing
        | invalid_flag
        | implausible_value
    )

    missing_count += has_missing.astype("int8")
    err_count += is_err.astype("int8")
    quality_issue_count += quality_issue.astype("int8")

    quality_report.append({
        "sensor": sensor,
        "missing_readings": int(has_missing.sum()),
        "explicit_ERR": int(is_err.sum()),
        "unexpected_missing": int(unexpected_missing.sum()),
        "invalid_flags": int(invalid_flag.sum()),
        "implausible_values": int(implausible_value.sum()),
    })

df["missing_sensor_count"] = missing_count
df["sensor_err_count"] = err_count
df["sensor_quality_issue_count"] = quality_issue_count

quality_summary = (
    pd.DataFrame(quality_report)
    .set_index("sensor")
)

print("\nSensor-quality summary:")
print(quality_summary)

print("\nRows containing unexpected quality issues:")
print((df["sensor_quality_issue_count"] > 0).sum())


Sensor-quality summary:
        missing_readings  explicit_ERR  unexpected_missing  invalid_flags  \
sensor                                                                      
s1                   155           153                   2              2   
s2                   103           102                   1              1   
s3                  1201          1199                   2              2   
s4                     3             2                   1              1   

        implausible_values  
sensor                      
s1                       3  
s2                       1  
s3                       0  
s4                       0  

Rows containing unexpected quality issues:
9
